<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/FastAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FastAPI Homepage: https://github.com/fastapi/fastapi

# Install dependencies

In [ ]:
!!pip install fastapi uvicorn nest_asyncio

# Create and run the app

In [ ]:
# app.py
import nest_asyncio
import uvicorn
from fastapi import FastAPI
import threading

# Patch event loop so uvicorn can run inside Colab's existing loop
nest_asyncio.apply()

app = FastAPI()

# GET — retrieve data, no side effects, params go in the URL
# Example: http://localhost:8000/
@app.get("/")
def root():
    return {"message": "Hello from FastAPI on Colab!"}

# GET with query parameter — passed as ?formula=GaN in the URL
# Example: http://localhost:8000/predict?formula=GaN
@app.get("/predict")
def predict(formula: str = "Si"):
    return {"formula": formula, "predicted_property": 42.0}

# POST — send data in the request body (JSON), used for creating/submitting
# Example: curl -X POST ... -d '{"formula":"GaN"}'
# @app.post("/predict_post")

# PUT — update/replace an existing resource entirely
# PATCH — partially update an existing resource
# DELETE — remove a resource

# Start server in background so the notebook stays interactive
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

# Visit http://localhost:8000/docs for auto-generated Swagger UI

| Method     | Purpose          | Data in... | Example use                    |
|------------|------------------|------------|--------------------------------|
| **GET**    | Read/fetch data  | URL params | Look up a prediction           |
| **POST**   | Create/submit    | JSON body  | Submit a structure for analysis |
| **PUT**    | Replace entirely | JSON body  | Overwrite a saved model        |
| **PATCH**  | Partial update   | JSON body  | Update one field               |
| **DELETE** | Remove           | URL params | Delete a saved result          |

# Test it locally

In [ ]:
import requests

r = requests.get("http://localhost:8000/")
print(r.json())

r = requests.get("http://localhost:8000/predict?formula=GaN")
print(r.json())

| Flag | Meaning | Example |
|------|---------|---------|
| `curl` | Command-line tool for making HTTP requests | `curl http://localhost:8000/` |
| `-s` | Silent mode — hides progress bar/stats | Without it you get download % noise |
| `-X` | Specify HTTP method (GET, POST, PUT, DELETE) | `-X POST`, `-X DELETE` |
| `-H` | Set a header | `-H "Content-Type: application/json"` |
| `-d` | Send data in the request body | `-d '{"formula": "GaN"}'` |
| `\| python3 -m json.tool` | Pipe output to pretty-print JSON | Makes response readable |

In [ ]:
# GET
!curl -s http://localhost:8000/ | python3 -m json.tool

In [ ]:
# GET with query param
!curl -s "http://localhost:8000/predict?formula=GaN" | python3 -m json.tool

See many apps like these with curl examples: https://atomgptlab.github.io/agapi/apps/

In [ ]:
!pip freeze | tee requirements.txt

## Deploying FastAPI to Production

### Local Development → Production Checklist

1. **Organize your code**
   - `app.py` — your FastAPI app
   - `requirements.txt` — dependencies (`pip freeze > requirements.txt`)

2. **Don't use `uvicorn.run()` in code** — run from command line instead:
```bash
   uvicorn app:app --host 0.0.0.0 --port 8000 --workers 4
```

3. **Common deployment options (easiest → most control):**

| Platform | Cost | Difficulty | Best for |
|----------|------|------------|----------|
| **HuggingFace Spaces** | Free | Easy | ML demos, class projects |
| **Render** | Free tier | Easy | Small apps, prototypes |
| **Railway** | Free tier | Easy | Quick deploys from GitHub |
| **Google Cloud Run** | Pay per use | Medium | Scalable, serverless |
| **AWS EC2 / GCP VM** | Pay per use | Hard | Full control, GPU support |

### Minimal Deploy Example (HuggingFace Spaces)

1. Create a repo on huggingface.co/spaces (select "Docker" SDK)
2. Add `Dockerfile`:
```dockerfile
   FROM python:3.10
   WORKDIR /app
   COPY requirements.txt .
   RUN pip install -r requirements.txt
   COPY app.py .
   CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]
```
3. Push code → auto-deploys with a public URL

### Key Production Additions
- **CORS** — allow frontend apps to call your API (`pip install fastapi[all]`)
- **HTTPS** — handled automatically by most platforms above
- **Auth** — add API keys or OAuth for protected endpoints
- **Logging** — track requests, errors, latency
- **Rate limiting** — prevent abuse